# 01 · 解析 PubMed 数据（nbib）

- 直接解析 `data/raw/*.nbib`（**不经过 Zotero**）
- 合并、按 PMID 去重
- 输出干净的 `data/interim/articles.csv`


## 第一步：读取数据

找到 `data/raw/` 下的所有 `.nbib` 原始文件。

In [ ]:
import os
import glob

# --- 原始数据目录 ---
# notebook 位于 notebooks/，数据在上一级的 data/raw/
raw_dir = os.path.join(os.getcwd(), '..', 'data', 'raw')

# --- 找到所有 nbib 文件 ---
nbib_files = sorted(glob.glob(os.path.join(raw_dir, 'pubmed-XJTU-*.nbib')))

print(f"在 {raw_dir} 找到 {len(nbib_files)} 个 nbib 文件：")
for f in nbib_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f"  {os.path.basename(f):28s}  {size_mb:6.1f} MB")


## 第二步：检查原始数据有哪些字段

`nbib`（MEDLINE 格式）是 `标签 - 值` 的纯文本，字段是隐藏的。
先把一个文件里出现的所有字段标签列出来，看看有哪些可用元数据。

In [ ]:
from collections import Counter

# 拿第一个文件做代表，扫描它出现的所有字段标签
sample_file = nbib_files[0]

tag_counts = Counter()
tag_sample = {}
with open(sample_file, encoding='utf-8') as f:
    for line in f:
        # 字段行形如 'TAG  - value'；续行以空格开头，跳过
        if len(line) > 6 and line[0] != ' ' and '-' in line[:6]:
            tag = line[:4].strip()
            value = line[6:].strip()
            tag_counts[tag] += 1
            tag_sample.setdefault(tag, value[:45])

print(f"{os.path.basename(sample_file)} 中共出现 {len(tag_counts)} 种字段：")
print()
print(f"{'标签':<6}{'次数':>7}   示例")
for tag, cnt in tag_counts.most_common():
    print(f"{tag:<6}{cnt:>7}   {tag_sample.get(tag, '')}")

## 字段说明与建议

### 为什么每种标签的出现次数不一样？

有两个原因（以 2021 年 4,563 篇为例）：

**原因 1 · 有些标签一篇论文会重复多次**
- `AD`（单位）出现 44,853 次 —— 一篇文章有几个作者就有几个单位
- `AU` / `FAU`（作者）38,604 次 —— 一篇多作者就多行
- `MH`（主题词）31,797 次 —— 一篇能标十几个 MeSH 词
- 这类「重复字段」的总次数会远超论文数。

**原因 2 · 有些标签不是每篇都有（可选项）**
- `MH` 只覆盖 64.9% —— 部分记录还没被 MEDLINE 标引
- `AB`（摘要）覆盖 96.6% —— 157 篇没有摘要
- `AUID`（ORCID）覆盖 43.1% —— 只有注册 ORCID 的作者才有
- `GR`（基金）21.3%、`PMC` 51.5%、`OT`（作者关键词）78%
- `VI / IP / PG`（卷期页）—— 电子优先发表的文章还没分卷期页

> 结论：`PMID` 这类「单值必备字段」每篇都有（= 4,563）；`AU`/`MH` 这类「重复字段」一篇多行；`AB`/`AUID` 这类「可选项」不是每篇都有。

### 🟢 核心字段（最终保留这 12 个）

| 标签 | 中文含义 | 示例 | 覆盖 |
|---|---|---|---|
| `PMID` | PubMed 编号 | `33823184` | 100% |
| `TI` | 标题 | `Melanocortin receptor-4…` | 100% |
| `AB` | 摘要 | `Glucagon-like peptide-2…` | 96.6% |
| `FAU` | 作者全名 | `Sun, Huiling` | 100% |
| `AD` | 作者单位 | `Key Laboratory of Shaanxi…` | 100% |
| `AUID` | 作者标识符（ORCID） | `ORCID: 0000-0002-…` | 43.1% |
| `MH` | MeSH 主题词（受控词） | `Animals` | 64.9% |
| `OT` | 作者关键词（自由词） | `Exendin` | 78.0% |
| `PT` | 出版类型 | `Journal Article` | 100% |
| `DP` | 出版日期 | `2021 Jun 15` | 100% |
| `JT` | 期刊全名 | `European journal of pharmacology` | 100% |
| `LID` | 位置标识（含 DOI） | `…[doi]` | 99.3% |

> 注：DOI 也可能出现在 `AID`（文章标识符）里，解析时作为 `LID` 的备用来源一起找。

### 🟡 可选字段

| 标签 | 中文含义 | 示例 | 覆盖 | 建议 |
|---|---|---|---|---|
| `AU` | 作者缩写名 | `Sun H` | 100% | △ 备选（不如 FAU 全） |
| `AID` | 文章标识符（含 DOI/PII） | `S0014-…[pii]` | 99.4% | △ DOI 备用来源 |
| `LA` | 语言 | `eng` | 100% | △ 筛英文 |
| `IS` | ISSN | `1879-0712 (Electronic)` | 100% | ✗ |
| `VI` | 卷 | `901` | 99.7% | △ |
| `IP` | 期 | `5` | 71.2% | △ |
| `PG` | 页码 | `174072` | 91.3% | △ |
| `DEP` | 电子出版日期 | `20210403` | 87.4% | △ |
| `SB` | 期刊子集 | `IM` | 76.4% | △ |
| `GR` | 基金资助 | `81701869/National…` | 21.3% | △ 基金分析 |
| `RN` | 化学物质登记号 | `0 (Appetite Depressants)` | 43.5% | △ |
| `PMC` | PMC 编号 | `PMC7981806` | 51.5% | △ 开放获取 |
| `SI` | 次要来源 ID（试验注册号） | `ClinicalTrials.gov/NCT…` | 1.3% | △ |
| `COIS` | 利益冲突声明 | `Declaration of Competing…` | 52.9% | △ |
| `CN` | 团体作者 | `STEP Study Group` | 0.6% | △ |
| `FIR` | 研究者全名（团体） | `Cai, Jun` | 0.2% | △ |
| `PHST` | 出版流程状态日期 | `2021/02/25 [received]` | 100% | ✗ |
| `PST` | 出版状态 | `ppublish` | 100% | ✗ |
| `EDAT` | 入库日期 | `2021/04/07` | 100% | ✗ |

### ⚪ 一般忽略

`OWN`(记录所有者) · `STAT`(记录状态) · `LR`(最后修订) · `PL`(出版国家) · `JID`(期刊ID) · `MHDA`(MeSH标引日期) · `CRDT`(创建日期) · `SO`(来源引用串) · `OTO`(OT所有者) · `DCOM`(完成日期) · `CI`(版权) · `PMCR`(PMC发布日期) · `IR`(研究者缩写) · `IRAD`(研究者单位) · `MID`(稿号) · `CIN`(被评论) · `CON`(评论对象) · `EIN`(被勘误) · `EFR`(勘误对象) · `RIN`(撤稿来源) · `ROF`(撤稿对象) · `ECI`(关注声明) · `RRI`(再撤稿相关) · `UOF`(更新对象) · `TT`(原文标题) · `OAB`(其他摘要) · `OABL`(其他摘要语言) · `PS`(作为主题的人名) · `FPS`(作为主题的人名全名)

### ✅ 结论

解析器保留这 **12 个核心字段**：`PMID`、`TI`、`AB`、`FAU`、`AD`、`AUID`、`MH`、`OT`、`PT`、`DP`、`JT`、`LID`。
